# ADA Lab Assignment 2 - Algorithm Strategies
### Divide & Conquer | Greedy | Dynamic Programming | TSP
---

In [ ]:
# ── Imports ──────────────────────────────────────────────────────────────────
import time
import random
import itertools
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

plt.rcParams.update({
    'figure.dpi': 120,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': True,
    'grid.alpha': 0.3,
    'font.size': 11
})

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
print('All libraries loaded ✓')

---
## Task 1 — Divide and Conquer
### Algorithms: Merge Sort & Binary Search

**Divide and Conquer** splits a problem into smaller sub-problems, solves each recursively, then combines results.

| Algorithm     | Time Complexity        | Space |
|---------------|------------------------|-------|
| Merge Sort    | O(n log n) all cases   | O(n)  |
| Binary Search | O(log n)               | O(1)  |

In [ ]:
# ── Merge Sort ────────────────────────────────────────────────────────────────
def merge_sort(arr):
    """Recursively splits array in half and merges sorted halves."""
    if len(arr) <= 1:
        return arr
    mid = len(arr) // 2
    left  = merge_sort(arr[:mid])
    right = merge_sort(arr[mid:])
    return merge(left, right)

def merge(left, right):
    """Merge two sorted lists into one sorted list."""
    result, i, j = [], 0, 0
    while i < len(left) and j < len(right):
        if left[i] <= right[j]:
            result.append(left[i]); i += 1
        else:
            result.append(right[j]); j += 1
    result.extend(left[i:])
    result.extend(right[j:])
    return result

# ── Binary Search ─────────────────────────────────────────────────────────────
def binary_search(arr, target):
    """Search sorted array by repeatedly halving the search space."""
    lo, hi = 0, len(arr) - 1
    while lo <= hi:
        mid = (lo + hi) // 2
        if arr[mid] == target:
            return mid
        elif arr[mid] < target:
            lo = mid + 1
        else:
            hi = mid - 1
    return -1

# ── Quick correctness check ───────────────────────────────────────────────────
sample = [38, 27, 43, 3, 9, 82, 10]
sorted_sample = merge_sort(sample)
print('Input :', sample)
print('Sorted:', sorted_sample)
idx = binary_search(sorted_sample, 43)
print(f'Binary search for 43 → index {idx}  (value={sorted_sample[idx]})')

In [ ]:
# ── Performance measurement ───────────────────────────────────────────────────
sizes_dc = [500, 1000, 2000, 5000, 10000, 20000]
ms_times, bs_times = [], []

for n in sizes_dc:
    data = [random.randint(0, n*10) for _ in range(n)]

    t0 = time.perf_counter()
    sorted_data = merge_sort(data)
    ms_times.append(time.perf_counter() - t0)

    target = sorted_data[n // 2]
    t0 = time.perf_counter()
    for _ in range(1000):          # repeat to get measurable time
        binary_search(sorted_data, target)
    bs_times.append((time.perf_counter() - t0) / 1000)

print(f'{'Size':>8}  {'Merge Sort (ms)':>17}  {'Binary Search (µs)':>19}')
print('-' * 50)
for n, ms, bs in zip(sizes_dc, ms_times, bs_times):
    print(f'{n:>8}  {ms*1e3:>17.3f}  {bs*1e6:>19.3f}')

In [ ]:
# ── Plot: Divide and Conquer timing ──────────────────────────────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))

ax1.plot(sizes_dc, [t*1e3 for t in ms_times], 'o-', color='#4C72B0', lw=2, label='Measured')
ref = [n * np.log2(n) * ms_times[-1] / (sizes_dc[-1] * np.log2(sizes_dc[-1])) * 1e3 for n in sizes_dc]
ax1.plot(sizes_dc, ref, '--', color='#DD8452', lw=1.5, label='O(n log n) ref')
ax1.set(title='Merge Sort — Time vs Input Size', xlabel='Input Size (n)', ylabel='Time (ms)')
ax1.legend()

ax2.plot(sizes_dc, [t*1e6 for t in bs_times], 's-', color='#55A868', lw=2, label='Measured')
ref2 = [np.log2(n) * bs_times[-1] / np.log2(sizes_dc[-1]) * 1e6 for n in sizes_dc]
ax2.plot(sizes_dc, ref2, '--', color='#DD8452', lw=1.5, label='O(log n) ref')
ax2.set(title='Binary Search — Time vs Input Size', xlabel='Input Size (n)', ylabel='Time (µs)')
ax2.legend()

plt.tight_layout()
plt.savefig('images/task1_divide_conquer.png', bbox_inches='tight')
plt.show()
print('Plot saved ✓')

**Observation (Task 1):** Merge Sort's measured time closely follows the O(n log n) theoretical curve. Binary Search grows logarithmically — even at 20 000 elements it takes only ~3 µs because it needs at most log₂(20000) ≈ 15 comparisons.

---
## Task 2 — Sorting Performance Comparison
### Bubble Sort vs Merge Sort vs Quick Sort

| Algorithm    | Best     | Average  | Worst    | Stable |
|--------------|----------|----------|----------|--------|
| Bubble Sort  | O(n)     | O(n²)    | O(n²)    | Yes    |
| Merge Sort   | O(n log n)| O(n log n)| O(n log n)| Yes  |
| Quick Sort   | O(n log n)| O(n log n)| O(n²)   | No     |

In [ ]:
# ── Sorting algorithms ────────────────────────────────────────────────────────
def bubble_sort(arr):
    """Repeatedly swaps adjacent elements if out of order. O(n²)."""
    a = arr[:]
    n = len(a)
    for i in range(n):
        swapped = False
        for j in range(n - i - 1):
            if a[j] > a[j+1]:
                a[j], a[j+1] = a[j+1], a[j]
                swapped = True
        if not swapped:      # early exit if already sorted
            break
    return a

def quick_sort(arr):
    """Partition around pivot; recursively sort sub-arrays."""
    if len(arr) <= 1:
        return arr
    pivot = arr[len(arr) // 2]
    left   = [x for x in arr if x < pivot]
    middle = [x for x in arr if x == pivot]
    right  = [x for x in arr if x > pivot]
    return quick_sort(left) + middle + quick_sort(right)

# ── Benchmark ─────────────────────────────────────────────────────────────────
sizes_sort = [100, 500, 1000, 2000, 3000, 5000]
bubble_t, merge_t, quick_t = [], [], []

for n in sizes_sort:
    data = [random.randint(0, n*5) for _ in range(n)]

    t0 = time.perf_counter(); bubble_sort(data);   bubble_t.append(time.perf_counter()-t0)
    t0 = time.perf_counter(); merge_sort(data);    merge_t.append(time.perf_counter()-t0)
    t0 = time.perf_counter(); quick_sort(data);    quick_t.append(time.perf_counter()-t0)

print(f"{'Size':>6}  {'Bubble(ms)':>12}  {'Merge(ms)':>11}  {'Quick(ms)':>11}")
print('-'*46)
for n, b, m, q in zip(sizes_sort, bubble_t, merge_t, quick_t):
    print(f"{n:>6}  {b*1e3:>12.2f}  {m*1e3:>11.3f}  {q*1e3:>11.3f}")

In [ ]:
# ── Plot: Sorting comparison ──────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(sizes_sort, [t*1e3 for t in bubble_t], 'o-', color='#C44E52', lw=2, label='Bubble Sort  O(n²)')
ax.plot(sizes_sort, [t*1e3 for t in merge_t],  's-', color='#4C72B0', lw=2, label='Merge Sort   O(n log n)')
ax.plot(sizes_sort, [t*1e3 for t in quick_t],  '^-', color='#55A868', lw=2, label='Quick Sort   O(n log n) avg')
ax.set(title='Sorting Algorithm Comparison', xlabel='Input Size (n)', ylabel='Time (ms)')
ax.legend()
plt.tight_layout()
plt.savefig('images/task2_sorting_comparison.png', bbox_inches='tight')
plt.show()
print('Plot saved ✓')

**Observation (Task 2):** Bubble Sort becomes drastically slower as n grows (quadratic), while Merge Sort and Quick Sort stay nearly flat on this scale. Quick Sort is usually fastest in practice due to better cache locality, despite the same asymptotic complexity as Merge Sort.

---
## Task 3 — Greedy Algorithms
### A) Activity Selection  |  B) Fractional Knapsack

Greedy algorithms make the **locally optimal choice** at each step, hoping it leads to the global optimum.

| Problem              | Strategy                       | Optimal? |
|----------------------|--------------------------------|----------|
| Activity Selection   | Pick earliest finishing job    | Yes      |
| Fractional Knapsack  | Best value/weight ratio first  | Yes      |

In [ ]:
# ── 3A: Activity Selection ────────────────────────────────────────────────────
def activity_selection(activities):
    """
    Select max number of non-overlapping activities.
    activities: list of (start, end, name)
    Strategy  : sort by end time, greedily pick if start >= last end.
    """
    sorted_acts = sorted(activities, key=lambda x: x[1])
    selected = [sorted_acts[0]]
    last_end = sorted_acts[0][1]

    for act in sorted_acts[1:]:
        if act[0] >= last_end:
            selected.append(act)
            last_end = act[1]
    return selected

activities = [
    (1, 4, 'A'), (3, 5, 'B'), (0, 6, 'C'),
    (5, 7, 'D'), (3, 9, 'E'), (5, 9, 'F'),
    (6, 10, 'G'), (8, 11, 'H'), (8, 12, 'I'),
    (2, 14, 'J'), (12, 16, 'K')
]

result = activity_selection(activities)
print('All activities  :', [(n, s, e) for s, e, n in activities])
print('Selected        :', [(n, s, e) for s, e, n in result])
print(f'Max activities  : {len(result)} out of {len(activities)}')

In [ ]:
# ── 3B: Fractional Knapsack ───────────────────────────────────────────────────
def fractional_knapsack(items, capacity):
    """
    items   : list of (value, weight, name)
    capacity: max weight the knapsack can hold
    Strategy: sort by value/weight ratio descending; take as much as possible.
    Returns : (total_value, list of (name, fraction_taken))
    """
    sorted_items = sorted(items, key=lambda x: x[0]/x[1], reverse=True)
    total_value, remaining, taken = 0.0, capacity, []

    for value, weight, name in sorted_items:
        if remaining == 0:
            break
        fraction = min(1.0, remaining / weight)
        total_value += fraction * value
        remaining   -= fraction * weight
        taken.append((name, round(fraction, 4), round(fraction*weight, 2)))

    return round(total_value, 2), taken

items = [
    (60,  10, 'Item-A'),
    (100, 20, 'Item-B'),
    (120, 30, 'Item-C'),
]
capacity = 50

total, taken = fractional_knapsack(items, capacity)
print(f'Knapsack capacity: {capacity} kg')
print(f'\n{"Item":<10} {"Fraction":>10} {"Weight taken":>14}')
print('-' * 36)
for name, frac, wt in taken:
    print(f'{name:<10} {frac:>10.2%} {wt:>12.2f} kg')
print(f'\nTotal value: {total}')

In [ ]:
# ── Plot: Greedy visualizations ───────────────────────────────────────────────
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 7))

# Activity timeline
colors = {act[2]: ('#55A868' if act in result else '#C44E52') for act in activities}
for i, (s, e, n) in enumerate(activities):
    ax1.barh(i, e-s, left=s, color=colors[n], edgecolor='white', height=0.6)
    ax1.text((s+e)/2, i, n, ha='center', va='center', fontsize=8, color='white', fontweight='bold')
sel_patch   = mpatches.Patch(color='#55A868', label='Selected')
unsel_patch = mpatches.Patch(color='#C44E52', label='Not selected')
ax1.legend(handles=[sel_patch, unsel_patch], loc='upper right')
ax1.set(title='Activity Selection — Greedy Result', xlabel='Time', yticks=range(len(activities)),
        yticklabels=[f"{n}({s}-{e})" for s,e,n in activities])

# Knapsack bar
names = [t[0] for t in taken]
weights = [t[2] for t in taken]
bars = ax2.barh(names, weights, color=['#4C72B0','#DD8452','#55A868'][:len(taken)])
for bar, (name, frac, wt) in zip(bars, taken):
    ax2.text(wt/2, bar.get_y()+bar.get_height()/2,
             f'{frac:.0%}', ha='center', va='center', color='white', fontweight='bold')
ax2.axvline(capacity, color='red', linestyle='--', lw=1.5, label=f'Capacity={capacity}')
ax2.set(title='Fractional Knapsack — Weight Taken per Item', xlabel='Weight (kg)')
ax2.legend()

plt.tight_layout()
plt.savefig('images/task3_greedy.png', bbox_inches='tight')
plt.show()
print('Plot saved ✓')

**Observation (Task 3):** The greedy approach works optimally here because both problems have the **greedy choice property** and **optimal substructure**. Activity Selection picks the earliest-finishing task, freeing maximum future time. Fractional Knapsack picks by best value/weight ratio — fractions are allowed, so we never waste capacity.

---
## Task 4 — Dynamic Programming
### A) 0/1 Knapsack  |  B) Longest Common Subsequence (LCS)

DP solves problems with **overlapping subproblems** and **optimal substructure** by storing intermediate results.

| Problem       | Time       | Space  |
|---------------|------------|--------|
| 0/1 Knapsack  | O(n × W)   | O(n×W) |
| LCS           | O(m × n)   | O(m×n) |

In [ ]:
# ── 4A: 0/1 Knapsack ─────────────────────────────────────────────────────────
def knapsack_01(values, weights, capacity):
    """
    0/1 Knapsack via bottom-up DP.
    dp[i][w] = max value using first i items with capacity w.
    """
    n = len(values)
    dp = [[0] * (capacity + 1) for _ in range(n + 1)]

    for i in range(1, n + 1):
        for w in range(capacity + 1):
            dp[i][w] = dp[i-1][w]                         # skip item i
            if weights[i-1] <= w:                          # take item i
                dp[i][w] = max(dp[i][w],
                               dp[i-1][w - weights[i-1]] + values[i-1])
    # Traceback to find selected items
    selected, w = [], capacity
    for i in range(n, 0, -1):
        if dp[i][w] != dp[i-1][w]:
            selected.append(i-1)
            w -= weights[i-1]

    return dp[n][capacity], list(reversed(selected)), dp

values  = [60, 100, 120, 80, 50]
weights = [10,  20,  30, 25, 15]
capacity = 50

max_val, sel_idx, dp_table = knapsack_01(values, weights, capacity)
print(f'Items       : {list(range(len(values)))}')
print(f'Values      : {values}')
print(f'Weights     : {weights}')
print(f'Capacity    : {capacity}')
print(f'Selected    : items {sel_idx}')
print(f'Total value : {max_val}')
print(f'Total weight: {sum(weights[i] for i in sel_idx)}')

In [ ]:
# ── 4B: Longest Common Subsequence ───────────────────────────────────────────
def lcs(X, Y):
    """
    Find LCS of two strings using bottom-up DP table.
    dp[i][j] = length of LCS of X[:i] and Y[:j].
    """
    m, n = len(X), len(Y)
    dp = [[0]*(n+1) for _ in range(m+1)]

    for i in range(1, m+1):
        for j in range(1, n+1):
            if X[i-1] == Y[j-1]:
                dp[i][j] = dp[i-1][j-1] + 1
            else:
                dp[i][j] = max(dp[i-1][j], dp[i][j-1])

    # Traceback
    seq, i, j = [], m, n
    while i > 0 and j > 0:
        if X[i-1] == Y[j-1]:
            seq.append(X[i-1]); i -= 1; j -= 1
        elif dp[i-1][j] > dp[i][j-1]:
            i -= 1
        else:
            j -= 1

    return dp[m][n], ''.join(reversed(seq)), dp

X, Y = 'AGGTAB', 'GXTXAYB'
length, subseq, lcs_dp = lcs(X, Y)
print(f'String X     : {X}')
print(f'String Y     : {Y}')
print(f'LCS length   : {length}')
print(f'LCS          : {subseq}')

In [ ]:
# ── Plot: DP table heatmaps ───────────────────────────────────────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Knapsack DP table (items 0-5, capacity 0-50, show every 5th column)
step = 5
dp_arr = np.array(dp_table)[:, ::step]
im1 = ax1.imshow(dp_arr, cmap='Blues', aspect='auto')
ax1.set(title='0/1 Knapsack DP Table', xlabel='Capacity (×5)', ylabel='Item index')
plt.colorbar(im1, ax=ax1, label='Max Value')

# LCS DP table
lcs_arr = np.array(lcs_dp)
im2 = ax2.imshow(lcs_arr, cmap='Greens', aspect='auto')
ax2.set(title=f'LCS DP Table  ("{X}" vs "{Y}")',
        xlabel='Y index', ylabel='X index',
        xticks=range(len(Y)+1), xticklabels=['']+list(Y),
        yticks=range(len(X)+1), yticklabels=['']+list(X))
for i in range(lcs_arr.shape[0]):
    for j in range(lcs_arr.shape[1]):
        ax2.text(j, i, lcs_arr[i,j], ha='center', va='center', fontsize=9)
plt.colorbar(im2, ax=ax2, label='LCS Length')

plt.tight_layout()
plt.savefig('images/task4_dp.png', bbox_inches='tight')
plt.show()
print('Plot saved ✓')

In [ ]:
# ── DP vs Brute Force time comparison (Knapsack) ─────────────────────────────
import math

def knapsack_brute(values, weights, capacity):
    """Try all 2^n subsets. Exponential — only feasible for small n."""
    n, best = len(values), 0
    for mask in range(1 << n):
        tv = tw = 0
        for i in range(n):
            if mask & (1 << i):
                tv += values[i]; tw += weights[i]
        if tw <= capacity:
            best = max(best, tv)
    return best

sizes_dp, dp_t_list, bf_t_list = list(range(2, 18)), [], []
cap = 50

for n in sizes_dp:
    vs = [random.randint(10, 100) for _ in range(n)]
    ws = [random.randint(5,  30)  for _ in range(n)]

    t0 = time.perf_counter(); knapsack_01(vs, ws, cap);    dp_t_list.append(time.perf_counter()-t0)
    t0 = time.perf_counter(); knapsack_brute(vs, ws, cap); bf_t_list.append(time.perf_counter()-t0)

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(sizes_dp, [t*1e3 for t in bf_t_list], 'o-', color='#C44E52', lw=2, label='Brute Force O(2ⁿ)')
ax.plot(sizes_dp, [t*1e3 for t in dp_t_list], 's-', color='#4C72B0', lw=2, label='DP  O(n×W)')
ax.set(title='0/1 Knapsack: DP vs Brute Force', xlabel='Number of Items (n)', ylabel='Time (ms)')
ax.legend()
plt.tight_layout()
plt.savefig('images/task4_dp_vs_brute.png', bbox_inches='tight')
plt.show()
print('Plot saved ✓')

**Observation (Task 4):** DP trades memory for time — the table stores solutions to all sub-problems, avoiding recomputation. For Knapsack with n=17 items the brute-force checks 131 072 subsets while DP only fills a 17×51 table. The difference grows exponentially.

---
## Task 5 — Travelling Salesman Problem (TSP)
### Brute-force exact solution — demonstrates exponential growth

**TSP:** Given n cities and distances between them, find the shortest route that visits every city exactly once and returns to the start.

| n cities | Permutations | Complexity |
|----------|-------------|------------|
| 4        | 6           | O(n!)      |
| 6        | 60          | O(n!)      |
| 8        | 2 520       | O(n!)      |
| 10       | 181 440     | O(n!)      |

In [ ]:
# ── TSP: Brute Force ──────────────────────────────────────────────────────────
def generate_distance_matrix(n, seed=42):
    """Create a symmetric random distance matrix for n cities."""
    rng = np.random.default_rng(seed)
    coords = rng.integers(0, 100, size=(n, 2))
    dist = np.zeros((n, n))
    for i in range(n):
        for j in range(n):
            dist[i][j] = np.hypot(coords[i][0]-coords[j][0],
                                  coords[i][1]-coords[j][1])
    return dist, coords

def tsp_brute_force(dist):
    """Try all (n-1)! permutations and return the shortest route."""
    n = len(dist)
    cities = list(range(1, n))       # fix city 0 as start
    best_cost, best_route = float('inf'), None

    for perm in itertools.permutations(cities):
        route = (0,) + perm + (0,)
        cost  = sum(dist[route[i]][route[i+1]] for i in range(n))
        if cost < best_cost:
            best_cost, best_route = cost, route

    return best_cost, best_route

# Demonstrate on 5 cities
dist5, coords5 = generate_distance_matrix(5)
cost5, route5  = tsp_brute_force(dist5)
print(f'5-city TSP → Best cost: {cost5:.2f}')
print(f'Route: {" → ".join(map(str, route5))}')

In [ ]:
# ── TSP: Measure exponential growth ──────────────────────────────────────────
tsp_sizes  = [4, 5, 6, 7, 8, 9, 10]
tsp_times  = []

for n in tsp_sizes:
    dist_m, _ = generate_distance_matrix(n)
    t0 = time.perf_counter()
    tsp_brute_force(dist_m)
    tsp_times.append(time.perf_counter() - t0)
    print(f'n={n:2d}  permutations={math.factorial(n-1):>8,}  time={tsp_times[-1]*1e3:8.3f} ms')

In [ ]:
# ── Plot: TSP route + exponential growth ─────────────────────────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

# Optimal route map
dist5, coords5 = generate_distance_matrix(5)
cost5, route5  = tsp_brute_force(dist5)
for i in range(len(route5)-1):
    c1, c2 = coords5[route5[i]], coords5[route5[i+1]]
    ax1.annotate('', xy=c2, xytext=c1,
                 arrowprops=dict(arrowstyle='->', color='#4C72B0', lw=2))
ax1.scatter(coords5[:,0], coords5[:,1], s=150, zorder=5, color='#C44E52')
for i, (x,y) in enumerate(coords5):
    ax1.annotate(f'  C{i}', (x, y), fontsize=11, fontweight='bold')
ax1.set(title=f'TSP Optimal Route (5 cities, cost={cost5:.1f})',
        xlabel='X', ylabel='Y')

# Exponential growth
ax2.semilogy(tsp_sizes, tsp_times, 'o-', color='#C44E52', lw=2.5, markersize=8)
ax2.set(title='TSP Brute Force — Exponential Growth', xlabel='Number of Cities (n)',
        ylabel='Time (s, log scale)')
for x, y in zip(tsp_sizes, tsp_times):
    ax2.annotate(f'{y*1e3:.2f}ms', (x, y), textcoords='offset points',
                 xytext=(5, 3), fontsize=8)

plt.tight_layout()
plt.savefig('images/task5_tsp.png', bbox_inches='tight')
plt.show()
print('Plot saved ✓')

**Observation (Task 5):** TSP is NP-Hard. Brute-force evaluates (n-1)! routes — going from 9 to 10 cities multiplies permutations by 9×. For 20 cities this would be ~10¹⁸ routes, impossible to compute exactly. In practice, heuristics (nearest-neighbour, genetic algorithms) or DP with bitmask (Held-Karp, O(n²·2ⁿ)) are used.

---
## Summary Comparison Table

| Task | Algorithm | Paradigm | Time Complexity | Best Use Case |
|------|-----------|----------|-----------------|---------------|
| 1 | Merge Sort | Divide & Conquer | O(n log n) | Large-scale sorting |
| 1 | Binary Search | Divide & Conquer | O(log n) | Search in sorted data |
| 2 | Bubble Sort | Brute Force | O(n²) | Tiny or nearly-sorted data |
| 2 | Quick Sort | Divide & Conquer | O(n log n) avg | General-purpose fast sort |
| 3 | Activity Selection | Greedy | O(n log n) | Scheduling / resource booking |
| 3 | Fractional Knapsack | Greedy | O(n log n) | Cargo loading, portfolio mix |
| 4 | 0/1 Knapsack (DP) | Dynamic Programming | O(n×W) | Budget allocation |
| 4 | LCS | Dynamic Programming | O(m×n) | Diff tools, DNA alignment |
| 5 | TSP Brute Force | Exhaustive | O(n!) | Exact routing (tiny n only) |

## Reflection

- **Divide & Conquer** shines when problems can be cleanly split (sorting, searching). Recursion overhead is worth the O(n log n) payoff over O(n²).
- **Greedy** is fast and simple but only provably optimal for specific problem structures (Activity Selection, Fractional Knapsack). It fails for 0/1 Knapsack.
- **Dynamic Programming** solves problems that look exponential in O(n²) or O(n×W) by memoising overlapping sub-problems — a huge practical win.
- **TSP** exposes the limits of exact computation. Real-world delivery routing uses approximation algorithms and heuristics — theoretical complexity directly dictates engineering choices.